__Two-Layer MLP Simulations (MLP1)__

Alfred

A 2 --> 2 --> 1 network. The hidden layer is two STAR perceptrons; the output layer is a single
trans-splicing ribozyme perceptron, which uses the same system of ODEs as the General Model.

In [ ]:
# Libraries for mathematical analysis
import numpy as np
from skimage import measure
import sympy as sym
import pylab as pl
import matplotlib.pyplot as plt
import scipy.integrate
import biocircuits

# Libraries to visualize results
import bokeh.io
from bokeh.io import show
from bokeh.plotting import figure
from bokeh.plotting import column
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from bokeh.layouts import row

bokeh.io.output_notebook()

<u>MLP Model</u>

![MLP Reaction Scheme](MLP1_Reaction.png)

A two-layer network. Layer 1 holds two STAR perceptrons, $j$ and $k$; layer 2 holds a single
trans-splicing ribozyme perceptron. Each input plasmid $X_i$ drives **both** a STAR cassette
($\alpha^{+}$) and a sequestering-RNA cassette ($\alpha^{-}$) at each hidden node, so the effective
weight $w_{n,i} \propto \alpha^{+}_{n,i} - \alpha^{-}_{n,i}$ can take either sign.

__Wiring between the layers.__ The released transcripts of layer 1 *are* the input species of
layer 2 -- they are not re-transcribed from a plasmid. Node $j$'s output $Y^{(1)}_j$ plays the role
of the ribozyme's input RNA, and node $k$'s output $Y^{(1)}_k$ plays the role of its sequestering
RNA. Writing it this way (rather than composing two isolated activation functions) makes the
downstream loading/retroactivity terms appear automatically and avoids double counting.

__Layer 1 -- STAR perceptrons__ ($n \in \{j, k\}$, target $P^{(1)}_n$ is catalytic and conserved):
\begin{align*}
&\dot{m}^{(1)}_n = \textstyle\sum_i \alpha^{+}_{n,i} x_i + \alpha^{+}_{n,0} - \phi m^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + d^{(1)}_n c^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{z}^{(1)}_n = \textstyle\sum_i \alpha^{-}_{n,i} x_i + \alpha^{-}_{n,0} - \phi z^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{p}^{(1)}_n = d^{(1)}_n c^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + v^{(1)}_n c^{(1)}_n\\
&\dot{c}^{(1)}_n = a^{(1)}_n p^{(1)}_n m^{(1)}_n - d^{(1)}_n c^{(1)}_n - v^{(1)}_n c^{(1)}_n\\
&\dot{q}^{(1)}_n = b^{(1)}_n z^{(1)}_n m^{(1)}_n - u^{(1)}_n q^{(1)}_n - \phi q^{(1)}_n
\end{align*}

__Layer-1 outputs (= layer-2 inputs).__ These carry the extra terms for being consumed downstream:
\begin{align*}
&\dot{y}^{(1)}_j = v^{(1)}_j c^{(1)}_j - \phi y^{(1)}_j - a_2 p^{(2)} y^{(1)}_j + d_2 c^{(2)} - b_2 y^{(1)}_k y^{(1)}_j + u_2 q^{(2)}\\
&\dot{y}^{(1)}_k = v^{(1)}_k c^{(1)}_k - \phi y^{(1)}_k - b_2 y^{(1)}_k y^{(1)}_j + u_2 q^{(2)}
\end{align*}

__Layer 2 -- trans-splicing ribozyme perceptron__ (General Model kinetics):
\begin{align*}
&\dot{p}^{(2)} = \theta - \phi p^{(2)} - a_2 p^{(2)} y^{(1)}_j + d_2 c^{(2)}\\
&\dot{c}^{(2)} = a_2 p^{(2)} y^{(1)}_j - d_2 c^{(2)} - \phi c^{(2)} - v_2 c^{(2)}\\
&\dot{y}^{(2)} = v_2 c^{(2)} - \delta_2 y^{(2)}\\
&\dot{q}^{(2)} = b_2 y^{(1)}_k y^{(1)}_j - u_2 q^{(2)} - \phi q^{(2)}
\end{align*}

Where:
-   $X_1, X_2 = \text{Input plasmids (each carrying an } \alpha^{+} \text{ and an } \alpha^{-} \text{ cassette per hidden node)}$
-   $M^{(1)}_n = \text{STAR for hidden node } n$
-   $Z^{(1)}_n = \text{Sequestering RNA for hidden node } n$
-   $P^{(1)}_n = \text{Target RNA (paused elongation complex) of hidden node } n$
-   $C^{(1)}_n = \text{STAR-Target complex of hidden node } n$
-   $Y^{(1)}_n = \text{Released transcript of hidden node } n$
-   $Q^{(1)}_n = \text{Sequestered STAR of hidden node } n$
-   $P^{(2)} = \text{Ribozyme target RNA}$
-   $C^{(2)} = \text{Ribozyme-input complex}$
-   $Y^{(2)} = \text{Network output}$
-   $Q^{(2)} = \text{Sequestered layer-2 input } (Y^{(1)}_j \cdot Y^{(1)}_k)$

<u>Mathematically Derived Decision Boundaries & Activation Functions</u>

The closed-form activation functions below are a **composition** of the two single-node results:
layer 1 is evaluated with the STAR polynomial, its steady-state output flux ($\phi y^{(1)}_n = v_n c_n$)
is fed into the General Model polynomial for layer 2. This assumes the layers are **not** loading
each other, which the coupled ODEs further down do not assume.

> __Caveat.__ Treat this section as a sketch, not ground truth. The single-node polynomial in
> `GeneralModel_Simulations.ipynb` does not currently reproduce that notebook's own ODE
> simulation (see the "Analytic vs Simulated" check below), so the composition inherits that
> gap. The ODE sections are the reliable ones.

In [ ]:
### Define Parameters
# Layer 1 (STAR perceptrons) --> shared by hidden nodes j and k unless stated
a1   = 10.0   # STAR-target binding
d1   = 1.0    # unbinding
v1   = 1.0    # transcript release
b1   = 100.0  # sequestration binding
u1   = 0.2    # sequestration unbinding
phi  = 0.1    # RNA decay

# Layer 2 (trans-splicing ribozyme perceptron)
a2     = 10.0   # input-target binding
d2     = 1.0    # unbinding
v2     = 1.0    # output release
b2     = 100.0  # sequestration binding
u2     = 0.2    # sequestration unbinding
delta2 = 0.1    # output decay
theta  = 0.3    # ribozyme target transcription

# Target supplies
pTot1 = 0.5           # Layer-1 target (conserved, set by initial condition)
pTot2 = theta / phi   # Layer-2 target (set by theta/phi at steady state)


###starNodeOutput() --> Closed-form steady-state output of ONE isolated STAR perceptron
def starNodeOutput(prodM, prodZ, a, d, v, b, u, phi, p_tot):
    # prodM, prodZ are total transcription rates into m and z (already weight-summed over inputs)
    K1  = (d + v) / a
    K2  = (u + phi) / b
    tau = v / phi
    xi  = K2 / K1
    xi1 = phi * K1 / (v * p_tot)

    beta1 = prodM / (v * p_tot)
    beta2 = prodZ / (v * p_tot)

    A = (1 - xi)
    B = tau * (beta1 * xi - beta1 + beta2 + xi*xi1 + 2*xi - xi1 - 1)
    C = tau**2 * (beta1 - 2*beta1*xi - beta2 - xi - xi*xi1)
    D = beta1 * xi * tau**3

    roots = np.roots([A, B, C, D])
    realRoots = np.real(roots[np.isreal(roots)])
    valid = realRoots[(realRoots >= 0) & (realRoots <= tau)]
    y_n = np.max(valid) if len(valid) else 0.0
    return y_n * p_tot


###ribozymeOutput() --> Closed-form steady-state output of the layer-2 perceptron
def ribozymeOutput(fluxM, fluxZ, a, d, v, b, u, phi, delta, p_tot):
    # fluxM, fluxZ are the transcript FLUXES arriving from layer 1 (not concentrations)
    K1  = (phi + d + v) / a
    K2  = (u + phi) / b
    tau = delta / v
    xi  = K2 / K1
    xi1 = phi * K1 / ((phi + v) * p_tot)

    beta1 = fluxM / (v * p_tot)
    beta2 = fluxZ / (v * p_tot)

    A = (1 - xi)
    B = tau * (beta1 * xi - beta1 + beta2 + xi*xi1 + 2*xi - xi1 - 1)
    C = tau**2 * (beta1 - 2*beta1*xi - beta2 - xi - xi*xi1)
    D = beta1 * xi * tau**3

    roots = np.roots([A, B, C, D])
    realRoots = np.real(roots[np.isreal(roots)])
    valid = realRoots[(realRoots >= 0) & (realRoots <= tau)]
    y_n = np.max(valid) if len(valid) else 0.0
    return y_n * p_tot


###networkActivationFunction() --> Chains layer 1 into layer 2 for one (x1, x2) pair
def networkActivationFunction(x1, x2, weights):
    xin = np.array([x1, x2])

    # Weight-summed transcription into each hidden node (destination-first indexing: [node, input])
    prodM = weights["alphaPos"] @ xin + weights["biasPos"]
    prodZ = weights["alphaNeg"] @ xin + weights["biasNeg"]

    # Layer 1 --> isolated steady-state outputs of nodes j and k
    yj = starNodeOutput(prodM[0], prodZ[0], a1, d1, v1, b1, u1, phi, pTot1)
    yk = starNodeOutput(prodM[1], prodZ[1], a1, d1, v1, b1, u1, phi, pTot1)

    # Layer 2 --> node j is the input RNA, node k is the sequesterer. At steady state the flux
    # leaving an isolated node n is phi * y_n, which plays the role of alpha*x for layer 2.
    y2 = ribozymeOutput(phi * yj, phi * yk, a2, d2, v2, b2, u2, phi, delta2, pTot2)
    return y2, yj, yk

In [ ]:
### Network weights
# Destination-first indexing, matching the ML convention: alphaPos[n, i] is the edge INTO hidden
# node n FROM input i.  Row 0 = node j, row 1 = node k.  Column 0 = X1, column 1 = X2.
#
# biasPos / biasNeg are constitutive (input-independent) cassettes. They are ZERO here, which
# reproduces the reaction scheme exactly as drawn. See the final section for why a non-zero bias
# is needed before the network can separate anything a single perceptron cannot.
weightsDefault = {
    "alphaPos": np.array([[1.0, 0.6],    # node j: strong from X1, weaker from X2
                          [0.6, 1.0]]),  # node k: weaker from X1, strong from X2
    "alphaNeg": np.array([[0.2, 0.4],
                          [0.4, 0.2]]),
    "biasPos": np.zeros(2),
    "biasNeg": np.zeros(2),
}

### Plotting --> Analytic decision boundaries of the two hidden nodes and of the network output
x1Grid = np.linspace(0, 1, 120)
x2Grid = np.linspace(0, 1, 120)
X1g, X2g = np.meshgrid(x1Grid, x2Grid)

Yj_a = np.zeros_like(X1g)
Yk_a = np.zeros_like(X1g)
Y2_a = np.zeros_like(X1g)

for i in range(len(x1Grid)):
    for j in range(len(x2Grid)):
        y2, yj, yk = networkActivationFunction(X1g[j, i], X2g[j, i], weightsDefault)
        Y2_a[j, i], Yj_a[j, i], Yk_a[j, i] = y2, yj, yk


###plotAnalyticBoundary() --> Draws the level contour of an analytic surface in (X1, X2)
def plotAnalyticBoundary(ax, surface, level, x1Vals, x2Vals, color, label):
    contours = measure.find_contours(surface, level=level)
    if len(contours) == 0:
        print(f"{label}: never crosses {level:.3g} on this grid (max = {surface.max():.3g})")
        return
    for c, contour in enumerate(contours):
        xc = np.interp(contour[:, 1], np.arange(len(x1Vals)), x1Vals)
        yc = np.interp(contour[:, 0], np.arange(len(x2Vals)), x2Vals)
        ax.plot(xc, yc, color=color, linewidth=2.5, label=label if c == 0 else None)


fig, ax = plt.subplots(figsize=(7, 5))

# Hidden-node boundaries --> where each STAR perceptron switches on (half of its own max)
plotAnalyticBoundary(ax, Yj_a, 0.5 * Yj_a.max(), x1Grid, x2Grid, "tab:orange", "Hidden node $j$ ON")
plotAnalyticBoundary(ax, Yk_a, 0.5 * Yk_a.max(), x1Grid, x2Grid, "tab:green",  "Hidden node $k$ ON")

# Network output boundary
plotAnalyticBoundary(ax, Y2_a, 0.5 * Y2_a.max(), x1Grid, x2Grid, "tab:blue", "Network output ON")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel(r"$X_1$ (Input plasmid 1)")
ax.set_ylabel(r"$X_2$ (Input plasmid 2)")
ax.set_title("MLP Decision Boundaries (analytic composition)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
### Plotting --> Analytic activation function of the whole network
x2_vals = [0.2, 0.5, 1.0]
x1_vals = np.linspace(0, 1, 60)

plt.figure(figsize=(7, 5))
for x2 in x2_vals:
    y_vals = [networkActivationFunction(x1, x2, weightsDefault)[0] for x1 in x1_vals]
    plt.plot(x1_vals, y_vals, linewidth=2, label=rf"$X_2 = {x2}$")

plt.xlabel("X1 (Input plasmid 1)")
plt.ylabel(r"$y^{(2)}$ (Network output)")
plt.title("MLP Activation Function (analytic composition)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

<u>Simulated Decision Boundaries & Activation Functions</u>

In [ ]:
# Function to do the 2D projection of decision boundaries with normalization  <-- From the EBCP LATAM 2025 Workshop
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max())
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10)
    p.add_layout(cb, 'right')

    return ()

In [ ]:
###saveFigure() --> Export a Bokeh figure to SVG, falling back to HTML when no webdriver is found
#bokeh's export_svgs drives a headless browser, so it needs geckodriver (firefox) or chromedriver
#(chromium) on PATH. A Jupyter kernel launched from Anaconda Navigator / a desktop launcher does
#NOT inherit a PATH edited in ~/.zshrc or ~/.bash_profile -- that is the usual reason this works
#in a terminal but breaks inside the notebook.
import os
import shutil
import pathlib
from bokeh.embed import file_html
from bokeh.resources import CDN

#Directories holding geckodriver / chromedriver --> prepended to PATH for this kernel only.
#Uncomment or add entries as needed, then re-run this cell (no kernel restart required).
WEBDRIVER_PATHS = [
    # "/opt/homebrew/bin",
    # "/usr/local/bin",
    # "/opt/anaconda3/bin",
    # os.path.expanduser("~/.local/bin"),
]
for _p in WEBDRIVER_PATHS:
    if _p and _p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = _p + os.pathsep + os.environ.get("PATH", "")


#webdriverAvailable() --> True if bokeh can find a driver for SVG/PNG export
def webdriverAvailable():
    return any(shutil.which(driver) for driver in ("geckodriver", "chromedriver"))


#saveFigure() --> Never raises, so a missing webdriver cannot discard a long ODE run
def saveFigure(p, filename):
    pathlib.Path(filename).parent.mkdir(parents=True, exist_ok=True)
    p.output_backend = "svg"

    if webdriverAvailable():
        try:
            export_svgs(p, filename=filename)
            print(f"Saved {filename}")
            return filename
        except Exception as e:
            print(f"SVG export failed --> {type(e).__name__}: {e}")
    else:
        print("No geckodriver/chromedriver on PATH --> skipping SVG export")
        print("  Fix: conda install -c conda-forge firefox geckodriver")
        print("  Then add its directory to WEBDRIVER_PATHS above and re-run this cell.")

    #Fallback --> standalone HTML needs no webdriver and stays interactive
    htmlName = str(pathlib.Path(filename).with_suffix(".html"))
    with open(htmlName, "w", encoding="utf-8") as f:
        f.write(file_html(p, CDN, pathlib.Path(filename).stem))
    print(f"Saved {htmlName} instead")
    return htmlName


print("Webdriver available:", webdriverAvailable())

In [ ]:
#Function to help calculate the ODEs
#X --> Sets all concentrations to 0 at the start, except the layer-1 targets (conserved)
def mlpODEs(X, t, x1, x2, weights, rates):

    #Layer 1 node j, layer 1 node k, then layer 2
    (m_j, z_j, p_j, c_j, y_j, q_j,
     m_k, z_k, p_k, c_k, y_k, q_k,
     p_2, c_2, y_2, q_2) = X

    #Unpack rates from array
    phi = rates["phi"]          #Decay of RNA (shared)

    a_j = rates["a_j"]          #Binding of STAR & target, node j
    d_j = rates["d_j"]          #Dissociation of STAR & target, node j
    v_j = rates["v_j"]          #Release of transcript, node j
    b_j = rates["b_j"]          #Binding of STAR & sequesterer, node j
    u_j = rates["u_j"]          #Dissociation of STAR & sequesterer, node j

    a_k = rates["a_k"]          #Same, for node k
    d_k = rates["d_k"]
    v_k = rates["v_k"]
    b_k = rates["b_k"]
    u_k = rates["u_k"]

    theta   = rates["theta"]    #Production of ribozyme target
    a_2     = rates["a_2"]      #Binding of layer-2 input & ribozyme target
    d_2     = rates["d_2"]      #Dissociation
    v_2     = rates["v_2"]      #Release of output
    delta_2 = rates["delta_2"]  #Decay of output RNA
    b_2     = rates["b_2"]      #Binding of layer-2 input & sequesterer
    u_2     = rates["u_2"]      #Dissociation

    #Weight-summed transcription into each hidden node --> alphaPos/alphaNeg are [node, input]
    xin   = np.array([x1, x2])
    prodM = weights["alphaPos"] @ xin + weights["biasPos"]
    prodZ = weights["alphaNeg"] @ xin + weights["biasNeg"]

    #LAYER 1, NODE j --> STAR kinetics, target is catalytic so p_j + c_j is conserved
    dm_j = prodM[0] - phi*m_j - a_j*p_j*m_j + d_j*c_j - b_j*z_j*m_j + u_j*q_j
    dz_j = prodZ[0] - phi*z_j - b_j*z_j*m_j + u_j*q_j
    dp_j = d_j*c_j - a_j*p_j*m_j + v_j*c_j
    dc_j = a_j*p_j*m_j - d_j*c_j - v_j*c_j
    dq_j = b_j*z_j*m_j - u_j*q_j - phi*q_j

    #LAYER 1, NODE k
    dm_k = prodM[1] - phi*m_k - a_k*p_k*m_k + d_k*c_k - b_k*z_k*m_k + u_k*q_k
    dz_k = prodZ[1] - phi*z_k - b_k*z_k*m_k + u_k*q_k
    dp_k = d_k*c_k - a_k*p_k*m_k + v_k*c_k
    dc_k = a_k*p_k*m_k - d_k*c_k - v_k*c_k
    dq_k = b_k*z_k*m_k - u_k*q_k - phi*q_k

    #LAYER-1 OUTPUTS == LAYER-2 INPUTS
    #y_j is the ribozyme's input RNA --> consumed by gate binding and by sequestration with y_k
    dy_j = v_j*c_j - phi*y_j - a_2*p_2*y_j + d_2*c_2 - b_2*y_k*y_j + u_2*q_2
    #y_k is the ribozyme's sequestering RNA --> consumed only by sequestration
    dy_k = v_k*c_k - phi*y_k - b_2*y_k*y_j + u_2*q_2

    #LAYER 2 --> General Model kinetics
    dp_2 = theta - phi*p_2 - a_2*p_2*y_j + d_2*c_2
    dc_2 = a_2*p_2*y_j - d_2*c_2 - phi*c_2 - v_2*c_2
    dy_2 = v_2*c_2 - delta_2*y_2
    dq_2 = b_2*y_k*y_j - u_2*q_2 - phi*q_2

    return [dm_j, dz_j, dp_j, dc_j, dy_j, dq_j,
            dm_k, dz_k, dp_k, dc_k, dy_k, dq_k,
            dp_2, dc_2, dy_2, dq_2]


#Index of the network output y_2 in the state vector --> Used everywhere downstream
OUT = 14

Layer 1 ($n \in \{j,k\}$, STAR -- the target is catalytic so there is no $\phi$ on $c$):

$K_{C}^{(1)} = \frac{v^{(1)}_n + d^{(1)}_n}{a^{(1)}_n}\\$

$K_{Q}^{(1)} = \frac{\phi + u^{(1)}_n}{b^{(1)}_n}$

Layer 2 (ribozyme -- General Model):

$K_{C}^{(2)} = \frac{\phi + v_2 + d_2}{a_2}\\$

$K_{Q}^{(2)} = \frac{\phi + u_2}{b_2}$

In [ ]:
# Define a grid of input values (x1, x2)
x1 = np.linspace(0, 1, 50)
x2 = np.linspace(0, 1, 50)
X1, X2 = np.meshgrid(x1, x2)

#Activation function inputs
x1Vals = np.linspace(0, 1, 50)
x2Vals = [0.2, 0.5, 1.0]

# Simulation time --> long enough for the 16-species system to reach steady state
t = np.linspace(0, 300, 600)

#Initial species states --> layer-1 targets start at pTot1 because they are conserved
X0 = [0,     #m_j
      0,     #z_j
      pTot1, #p_j
      0,     #c_j
      0,     #y_j
      0,     #q_j
      0,     #m_k
      0,     #z_k
      pTot1, #p_k
      0,     #c_k
      0,     #y_k
      0,     #q_k
      0,     #p_2
      0,     #c_2
      0,     #y_2
      0]     #q_2

#Network weights --> reuse the ones defined in the analytic section
weights = weightsDefault

#Kinetics parameters --> theta is varied across variants (layer-2 target supply)
ratesDefault = {
    'phi':     np.array([0.1, 0.1, 0.1]),
    'a_j':     np.array([10, 10, 10]),
    'd_j':     np.array([1, 1, 1]),
    'v_j':     np.array([1, 1, 1]),
    'b_j':     np.array([100, 100, 100]),
    'u_j':     np.array([0.2, 0.2, 0.2]),
    'a_k':     np.array([10, 10, 10]),
    'd_k':     np.array([1, 1, 1]),
    'v_k':     np.array([1, 1, 1]),
    'b_k':     np.array([100, 100, 100]),
    'u_k':     np.array([0.2, 0.2, 0.2]),
    'theta':   np.array([0.3, 0.6, 0.9]),
    'a_2':     np.array([10, 10, 10]),
    'd_2':     np.array([1, 1, 1]),
    'v_2':     np.array([1, 1, 1]),
    'delta_2': np.array([0.1, 0.1, 0.1]),
    'b_2':     np.array([100, 100, 100]),
    'u_2':     np.array([0.2, 0.2, 0.2]),
}

#getVariantRates() --> Extracts variant-specific rates from ratesDefault
def getVariantRates(ratesArray, variantIndex):
    rates = {}
    for key, arr in ratesArray.items():
        rates[key] = arr[variantIndex]
    return rates

#getVariantLabel() --> Legend label (labelBy = "K_C2", "K_Q2", "theta", or "None" for no legend)
def getVariantLabel(rates, labelBy):
    if labelBy in ("None", None):
        return None
    if labelBy == "K_C2":
        return f"K_C2 = {(rates['phi'] + rates['v_2'] + rates['d_2']) / rates['a_2']:.3g}"
    if labelBy == "K_Q2":
        return f"K_Q2 = {(rates['phi'] + rates['u_2']) / rates['b_2']:.3g}"
    return f"theta = {rates['theta']:.3g}"

numVariants = len(ratesDefault['phi']) #phi is in all models --> Best choice

#Bokeh color palette for heatmaps + activation functions
colors = bokeh.palettes.OrRd3

#Array to store all results
results = []

In [ ]:
#Range through each variant --> Solve ODEs + Plot heatmap + Plot activation function --> Append to Results
#NOTE: 50x50 grid x 3 variants x a 16-species system --> this cell takes a few minutes
for variantIndex in range(numVariants):
    #Get rates for this current iteration's variant
    rates = getVariantRates(ratesDefault, variantIndex)

    #ODE SOLUTIONS
    #Array for storing scipy.integrate solutions
    gridSolutions = [[None for _ in range(len(x1))] for _ in range(len(x2))]

    # Initialize a matrix to store the steady-state output y
    Y = np.zeros_like(X1)

    # Iterate over the grid values for all variable inputs X1 and X2 --> Solve the ODEs
    for i in range(len(x1)):
        for j in range(len(x2)):
            solutions = scipy.integrate.odeint(mlpODEs, X0, t, args=(X1[j, i], X2[j, i], weights, rates))
            gridSolutions[j][i] = solutions
            Y[j, i] = solutions[-1, OUT] #Extract the steady state output y_2

    #HEATMAPS
    p_heatmap = figure(width=350, height=300, title=f"Variant {variantIndex}")

    #Make the heatmap
    contourf(
        p_heatmap,
        X1,
        X2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=True
    )

    p_heatmap.xaxis.axis_label = "X1"
    p_heatmap.yaxis.axis_label = "X2"

    #ACTIVATION FUNCTIONS
    activationFunctions = {}
    p_activation = figure(width=350, height=300, title=f"Variant {variantIndex}")

    for x2j, color in zip(x2Vals, colors):

        curves = []
        output_y = np.zeros_like(x1Vals)

        for i, x1i in enumerate(x1Vals):

            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(x1i, x2j, weights, rates)
            )

            curves.append(sol)
            output_y[i] = sol[-1, OUT]

        activationFunctions[x2j] = curves

        p_activation.line(
            x1Vals,
            output_y,
            line_width=3,
            color=color,
            legend_label=f"X2 = {x2j}"
        )

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    p_activation.legend.location = "top_left"

    #Append ODE solutions, heatmaps, and activation functions for all variants into the results array
    results.append({
        "variantIndex": variantIndex,
        "rates": rates,
        "solutions": {
            "grid": gridSolutions,
            "activation": activationFunctions
        },
        "heatmaps": p_heatmap,
        "activationFunctions": p_activation
    })

In [ ]:
#Plot results
for res in results:
    print(f"\nVariant {res['variantIndex']}")
    for k, v in res["rates"].items():
        print(f"{k}: {v}")

    show(row(res["heatmaps"], res["activationFunctions"]))

In [ ]:
###Sanity checks --> conservation, steady state, and analytic vs simulated
rates0 = getVariantRates(ratesDefault, 0)
sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(0.8, 0.4, weights, rates0))
final = sol[-1]

#Layer-1 targets are catalytic --> p_n + c_n must stay at pTot1
print(f"Conservation  p_j + c_j = {final[2] + final[3]:.6f}  (expected {pTot1})")
print(f"Conservation  p_k + c_k = {final[8] + final[9]:.6f}  (expected {pTot1})")

#All derivatives should be ~0 at the end of the integration window
residual = np.max(np.abs(mlpODEs(final, t[-1], 0.8, 0.4, weights, rates0)))
print(f"Max |dX/dt| at t_end  = {residual:.2e}  --> steady state reached if this is tiny")

#Analytic composition vs coupled ODEs --> expect a LARGE gap, see the caveat above
print("\n  x1   x2 |  ODE y_2   analytic y_2")
for xa in (0.2, 0.6, 1.0):
    for xb in (0.2, 0.6, 1.0):
        s = scipy.integrate.odeint(mlpODEs, X0, t, args=(xa, xb, weights, rates0))[-1, OUT]
        ana = networkActivationFunction(xa, xb, weights)[0]
        print(f"  {xa:.1f}  {xb:.1f} | {s:9.4f}   {ana:12.4f}")

In [ ]:
### plotActivationFunctionVariants() --> Plots the network activation function for every rate variant on the same Bokeh figure, holding X2 fixed, so the effect of the varied rate(s) is directly comparable
def plotActivationFunctionVariants(ratesDefault, x2_fixed, x1Vals, weights, X0, t, labelBy="None"):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    colors = bokeh.palettes.OrRd3  # Reuse the same palette as the rest of the notebook

    p_activation = figure(width=500, height=400, title=f"Network Activation Function Across Rate Variants")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        output_y = np.zeros_like(x1Vals)
        for i, x1i in enumerate(x1Vals):
            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(x1i, x2_fixed, weights, rates)
            )
            output_y[i] = sol[-1, OUT]  # Steady-state output y_2

        line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
        label = getVariantLabel(rates, labelBy)
        if label is not None:
            line_kwargs["legend_label"] = label  #labelBy="None" --> No legend

        p_activation.line(x1Vals, output_y, **line_kwargs)

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    if p_activation.legend:
        p_activation.legend.location = "top_left"

    show(p_activation)
    return p_activation

### Example call --> Hold X2 fixed at one of the existing x2Vals
p1 = plotActivationFunctionVariants(ratesDefault, x2_fixed=x2Vals[0], x1Vals=x1Vals, weights=weights, X0=X0, t=t, labelBy="theta")
saveFigure(p1, "Simulation_Plots/mlp1_activation_variants.svg")

In [ ]:
###plotDecisionBoundaryVariants() --> Extracts the Y = threshold contour (the decision boundary) for each rate variant and overlays them on one (X1, X2) plot
def plotDecisionBoundaryVariants(ratesDefault, threshold, x1, x2, X1, X2, weights, X0, t, relative=False, labelBy="None"):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    colors = bokeh.palettes.OrRd3

    p_boundary = figure(width=500, height=400, title=f"Decision Boundary Across Rate Variants (Y = {threshold}{' * max(Y)' if relative==True else ''})")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        # Solve the ODEs over the full (X1, X2) grid to get steady-state output Y
        Y = np.zeros_like(X1)
        Yj = np.zeros_like(X1)  # Steady-state output of hidden node j
        Yk = np.zeros_like(X1)  # Steady-state output of hidden node k
        for i in range(len(x1)):
            for j in range(len(x2)):
                sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(X1[j, i], X2[j, i], weights, rates))
                Y[j, i]  = sol[-1, OUT]  # Steady-state output y_2
                Yj[j, i] = sol[-1, 4]    # Steady-state free y_j
                Yk[j, i] = sol[-1, 10]   # Steady-state free y_k

        # relative=True --> threshold is a fraction of this variant's max Y
        level = threshold * Y.max() if relative else threshold

        # Extract the Y = level contour --> This is the decision boundary
        contours = measure.find_contours(Y, level=level)

        if len(contours) == 0:
            print(f"Variant {variantIndex}: Y never crosses {level:.3g} on this grid (max Y = {Y.max():.3g}) --> no boundary to plot")
            continue

        for c, contour in enumerate(contours):
            #find_contours returns [row_idx, col_idx] into the Y grid --> map indices back to real x1, x2 values
            x1_contour = np.interp(contour[:, 1], np.arange(len(x1)), x1)
            x2_contour = np.interp(contour[:, 0], np.arange(len(x2)), x2)

            line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
            if c == 0 and labelBy not in ("None", None):
                line_kwargs["legend_label"] = getVariantLabel(rates, labelBy)  #Only label once per variant

            p_boundary.line(x1_contour, x2_contour, **line_kwargs)

    p_boundary.xaxis.axis_label = "X1 (Input plasmid 1)"
    p_boundary.yaxis.axis_label = "X2 (Input plasmid 2)"
    if p_boundary.legend:
        p_boundary.legend.location = "top_left"

    show(p_boundary)
    return p_boundary

###threshold = Y value (fraction of max Y when relative=True) --> HELD CONSTANT
p2 = plotDecisionBoundaryVariants(ratesDefault, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, weights=weights, X0=X0, t=t, relative=True, labelBy="theta")
saveFigure(p2, "Simulation_Plots/mlp1_decision_boundary.svg")

In [ ]:
#3D-Plot of the network activation function
fig = go.Figure(data=[
    go.Surface(z=Y, x=X1, y=X2, colorscale="Viridis")
])

fig.update_layout(
    scene=dict(
        xaxis_title='Input plasmid 1 (X1)',
        yaxis_title='Input plasmid 2 (X2)',
        zaxis_title='Network Output (Y2)'
    ),
    width=800,
    height=600
)

fig.show()

<u>Why the Network Needs a Bias Cassette</u>

As drawn, every hidden node's production terms are **purely proportional** to the inputs:

$$\text{net}_n(x) = \sum_i \left(\alpha^{+}_{n,i} - \alpha^{-}_{n,i}\right) x_i$$

There is no constant term, so $\text{net}_n(0,0) = 0$ and **every hidden decision boundary passes
through the origin**. Two boundaries through the origin cannot carve out the XOR regions, so the
network collapses to something OR-like no matter how the $\alpha$'s are tuned.

The fix is a **constitutive (input-independent) cassette** at each hidden node -- a promoter with no
input dependence driving either the STAR or the sequestering RNA. That is the $\alpha^{\pm}_{n,0}$
term, and it is the biochemical realization of a neural-network bias:

$$\text{net}_n(x) = \sum_i \left(\alpha^{+}_{n,i} - \alpha^{-}_{n,i}\right) x_i + \left(\alpha^{+}_{n,0} - \alpha^{-}_{n,0}\right)$$

The cell below adds one constitutive sequestering-RNA cassette per hidden node, turning node $j$
into an OR gate and node $k$ into an AND gate. Since node $k$ inhibits the output, the network
computes OR AND-NOT AND = **XOR**.

In [ ]:
###XOR configuration --> adds a constitutive sequesterer cassette (the bias) at each hidden node
gain_k = 6.0   # Node k is driven harder so it can quench node j when BOTH inputs are high

weightsXOR = {
    # Both hidden nodes sum the two inputs equally
    "alphaPos": np.array([[1.0, 1.0],
                          [gain_k, gain_k]]),
    "alphaNeg": np.zeros((2, 2)),
    "biasPos": np.zeros(2),
    # Constitutive sequesterer --> sets each node's threshold
    #   node j fires once x1 + x2 > 0.5  --> OR
    #   node k fires once x1 + x2 > 1.5  --> AND
    "biasNeg": np.array([0.5, 1.5 * gain_k]),
}

#Layer-2 rates tuned so the sequestration between y_j and y_k wins decisively over gate binding
ratesXOR = {
    'phi': 0.1,
    'a_j': 10, 'd_j': 1, 'v_j': 1, 'b_j': 100, 'u_j': 0.2,
    'a_k': 10, 'd_k': 1, 'v_k': 1, 'b_k': 100, 'u_k': 0.2,
    'theta': 0.3,
    'a_2': 1.0,      # weaker gate binding
    'd_2': 1,
    'v_2': 1,
    'delta_2': 0.1,
    'b_2': 2000,     # much stronger layer-2 annihilation
    'u_2': 0.2,
}

### Truth table
print("  X1   X2  |   y_2")
truth = {}
for xa in (0.0, 1.0):
    for xb in (0.0, 1.0):
        sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(xa, xb, weightsXOR, ratesXOR))
        truth[(xa, xb)] = sol[-1, OUT]
        print(f"  {xa:.1f}  {xb:.1f}  |  {truth[(xa, xb)]:.4f}")

on  = min(truth[(0.0, 1.0)], truth[(1.0, 0.0)])
off = max(truth[(0.0, 0.0)], truth[(1.0, 1.0)])
print(f"\nON/OFF contrast = {on / (off + 1e-12):.1f}x")

### Heatmap of the XOR response surface
Yxor = np.zeros_like(X1)
for i in range(len(x1)):
    for j in range(len(x2)):
        sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(X1[j, i], X2[j, i], weightsXOR, ratesXOR))
        Yxor[j, i] = sol[-1, OUT]

p_xor = figure(width=400, height=350, title="XOR response surface")
contourf(p_xor, X1, X2, Yxor, palette=bokeh.palettes.Oranges8[::-1], normal=True)
p_xor.xaxis.axis_label = "X1"
p_xor.yaxis.axis_label = "X2"
show(p_xor)

fig_xor = go.Figure(data=[go.Surface(z=Yxor, x=X1, y=X2, colorscale="Viridis")])
fig_xor.update_layout(
    scene=dict(xaxis_title='X1', yaxis_title='X2', zaxis_title='Network Output (Y2)'),
    width=800, height=600, title="XOR"
)
fig_xor.show()